# Semana 26 — Agent-First Analytics: Structured Outputs para Agents
**Fase 5: MLOps, Deployment e Agent-First Products** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/05-Phase-5-MLOps-and-Agents/notebooks/semana_26_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-26-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/05-Phase-5-MLOps-and-Agents/docs/Semana-26-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

In [ ]:
import json, math
import numpy as np
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError
from scipy.stats import norm

## Passo 1 — Uma análise "para humanos"
O resultado de um teste A/B devolvido como texto. Tente escrever o código que um agente precisaria para decidir o que fazer a partir desta string.

In [ ]:
def analise_para_humanos(conv_a, n_a, conv_b, n_b):
    pa, pb = conv_a / n_a, conv_b / n_b
    return f"Grupo B converteu {pb:.1%} contra {pa:.1%} do A. Parece uma melhora de {(pb - pa) / pa:.0%}, provavelmente significativa."

print(analise_para_humanos(480, 10_000, 540, 10_000))

"Parece", "provavelmente": inutilizável por um programa.

## Passo 2 — O output schema
Defina `ResultadoAB` com os campos abaixo. Use `Field` para limites e descrições.

| Campo | Tipo |
|---|---|
| `versao_esquema` | `Literal["1.0"]`, padrão `"1.0"` |
| `metrica` | `str` |
| `taxa_controle`, `taxa_tratamento` | `float` entre 0 e 1 |
| `lift_relativo` | `float` |
| `ic95_diferenca` | `tuple[float, float]` (diferença absoluta) |
| `p_valor` | `float` entre 0 e 1 |
| `significativo` | `bool` |
| `recomendacao` | `Literal["lancar", "nao_lancar", "continuar_teste"]` |
| `avisos` | `list[str]`, padrão lista vazia |

In [ ]:
class ResultadoAB(BaseModel):
    versao_esquema: Literal["1.0"] = "1.0"
    metrica: str = Field(description="Nome da métrica analisada")
    taxa_controle: float = Field(ge=0, le=1)
    taxa_tratamento: float = Field(ge=0, le=1)
    lift_relativo: float = Field(description="(tratamento - controle) / controle")
    ic95_diferenca: tuple[float, float] = Field(description="IC de 95% da diferença absoluta entre as taxas")
    p_valor: float = Field(ge=0, le=1)
    significativo: bool
    recomendacao: Literal["lancar", "nao_lancar", "continuar_teste"]
    avisos: list[str] = Field(default_factory=list)
print(json.dumps(ResultadoAB.model_json_schema()["properties"]["recomendacao"], ensure_ascii=False))

## Passo 3 — A análise agent-first
Teste z de duas proporções. Regras de recomendação: significativo e lift > 0 → `lancar`; significativo e lift < 0 → `nao_lancar`; caso contrário → `continuar_teste`. Avisos: desbalanceamento entre grupos (razão fora de 0,9 a 1,1) e amostra pequena (algum grupo com menos de 1.000).

In [ ]:
class EntradaAB(BaseModel):
    metrica: str = "conversao"
    conversoes_controle: int = Field(ge=0); n_controle: int = Field(gt=0)
    conversoes_tratamento: int = Field(ge=0); n_tratamento: int = Field(gt=0)
    alfa: float = Field(default=0.05, gt=0, lt=1)

def analisar_ab(e: EntradaAB) -> ResultadoAB:
    pa, pb = e.conversoes_controle / e.n_controle, e.conversoes_tratamento / e.n_tratamento
    p_pool = (e.conversoes_controle + e.conversoes_tratamento) / (e.n_controle + e.n_tratamento)
    ep_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / e.n_controle + 1 / e.n_tratamento))
    z = (pb - pa) / ep_pool if ep_pool > 0 else 0.0
    p_valor = float(2 * (1 - norm.cdf(abs(z))))
    ep = math.sqrt(pa * (1 - pa) / e.n_controle + pb * (1 - pb) / e.n_tratamento)
    ic = (pb - pa - 1.96 * ep, pb - pa + 1.96 * ep)
    sig = p_valor < e.alfa; lift = (pb - pa) / pa if pa > 0 else 0.0
    rec = ("lancar" if lift > 0 else "nao_lancar") if sig else "continuar_teste"
    avisos = []
    if not 0.9 <= e.n_tratamento / e.n_controle <= 1.1: avisos.append("Grupos desbalanceados: verifique a randomização (SRM).")
    if min(e.n_controle, e.n_tratamento) < 1000: avisos.append("Amostra pequena: resultado instável.")
    return ResultadoAB(metrica=e.metrica, taxa_controle=pa, taxa_tratamento=pb, lift_relativo=lift, ic95_diferenca=ic,
                       p_valor=p_valor, significativo=sig, recomendacao=rec, avisos=avisos)

r = analisar_ab(EntradaAB(conversoes_controle=480, n_controle=10_000, conversoes_tratamento=540, n_tratamento=10_000))
print(r.model_dump_json(indent=1))

Agora a decisão do agente é uma linha: `if r.recomendacao == "lancar": ...`

## Passo 4 — Segunda tool: sample size
Para detectar um efeito relativo mínimo `mde` sobre uma `taxa_base`, com significância `alfa` e poder `poder`:

`n por grupo = (z_{1-α/2} + z_{poder})² · [p₁(1−p₁) + p₂(1−p₂)] / (p₂ − p₁)²`, com `p₂ = p₁ (1 + mde)`.

In [ ]:
class EntradaAmostra(BaseModel):
    taxa_base: float = Field(gt=0, lt=1, description="Taxa de conversão atual, entre 0 e 1 (ex.: 0.05 para 5%)")
    mde: float = Field(gt=0, description="Efeito relativo mínimo detectável (ex.: 0.1 para +10%)")
    alfa: float = Field(default=0.05, gt=0, lt=1); poder: float = Field(default=0.8, gt=0, lt=1)

class ResultadoAmostra(BaseModel):
    n_por_grupo: int; n_total: int; premissas: str

def tamanho_amostra(e: EntradaAmostra) -> ResultadoAmostra:
    p1, p2 = e.taxa_base, e.taxa_base * (1 + e.mde)
    z = norm.ppf(1 - e.alfa / 2) + norm.ppf(e.poder)
    n = math.ceil(z ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2)
    return ResultadoAmostra(n_por_grupo=n, n_total=2 * n, premissas=f"teste bilateral, alfa={e.alfa}, poder={e.poder}, alocação 50/50")

print(tamanho_amostra(EntradaAmostra(taxa_base=0.05, mde=0.10)))

## Passo 5 — Tool registry
Cada tool: descrição, modelo de entrada e função. `especificacao()` devolve o que seria enviado ao LLM.

In [ ]:
FERRAMENTAS = {
    "analisar_ab": {"descricao": "Analisa um teste A/B de proporções e devolve lift, IC95%, p-valor e uma recomendação.", "entrada": EntradaAB, "fn": analisar_ab},
    "tamanho_amostra": {"descricao": "Calcula o tamanho de amostra por grupo para um teste A/B de proporções.", "entrada": EntradaAmostra, "fn": tamanho_amostra},
}

def especificacao():
    return [{"nome": n, "descricao": f["descricao"], "parametros": f["entrada"].model_json_schema()} for n, f in FERRAMENTAS.items()]

print(json.dumps(especificacao()[1], ensure_ascii=False, indent=1)[:700])

## Passo 6 — Execução segura com structured errors
`executar(chamada_json)` recebe a string produzida pelo modelo. Deve devolver **sempre** um dicionário: `{"ok": True, "resultado": ...}` ou `{"ok": False, "erro": CÓDIGO, "detalhe": ..., "sugestao": ...}`. Códigos: `JSON_INVALIDO`, `FERRAMENTA_DESCONHECIDA`, `ARGUMENTO_INVALIDO`.

In [ ]:
def executar(chamada_json: str) -> dict:
    try:
        chamada = json.loads(chamada_json)
    except json.JSONDecodeError as e:
        return {"ok": False, "erro": "JSON_INVALIDO", "detalhe": str(e), "sugestao": 'Responda apenas com {"ferramenta": ..., "argumentos": {...}}'}
    nome = chamada.get("ferramenta")
    if nome not in FERRAMENTAS:
        return {"ok": False, "erro": "FERRAMENTA_DESCONHECIDA", "detalhe": str(nome), "sugestao": f"Use uma de: {list(FERRAMENTAS)}"}
    f = FERRAMENTAS[nome]
    try:
        entrada = f["entrada"](**chamada.get("argumentos", {}))
    except ValidationError as e:
        problemas = "; ".join(f"{'.'.join(map(str, x['loc']))}: {x['msg']}" for x in e.errors())
        return {"ok": False, "erro": "ARGUMENTO_INVALIDO", "detalhe": problemas, "sugestao": "Corrija os campos indicados conforme o esquema."}
    return {"ok": True, "resultado": f["fn"](entrada).model_dump()}

for c in ['{"ferramenta": "tamanho_amostra", "argumentos": {"taxa_base": 5, "mde": 0.1}}', '{"ferramenta": "prever_vendas"}', 'claro! vou calcular',
          '{"ferramenta": "tamanho_amostra", "argumentos": {"taxa_base": 0.05, "mde": 0.1}}']:
    print(executar(c))

## Passo 7 — O agent loop
O "LLM" aqui é uma função determinística que imita um modelo real, incluindo um erro típico (enviar 5 em vez de 0,05) e a correção depois de ler o erro. Implemente o loop: chamar o modelo, executar, anexar o resultado ao histórico, parar na resposta final ou ao atingir `max_passos`.

In [ ]:
def llm_simulado(historico):
    """Imita um LLM planejando um experimento. Devolve uma chamada de ferramenta (JSON) ou {"final": ...}."""
    obs = [h["conteudo"] for h in historico if h["papel"] == "ferramenta"]
    if not obs: return '{"ferramenta": "tamanho_amostra", "argumentos": {"taxa_base": 5, "mde": 0.1}}'
    if len(obs) == 1 and not obs[0]["ok"]: return '{"ferramenta": "tamanho_amostra", "argumentos": {"taxa_base": 0.05, "mde": 0.1}}'
    if len(obs) == 2: return '{"ferramenta": "analisar_ab", "argumentos": {"conversoes_controle": 1560, "n_controle": 31234, "conversoes_tratamento": 1702, "n_tratamento": 31234}}'
    ultimo = obs[-1]["resultado"]
    return json.dumps({"final": f"Recomendação: {ultimo['recomendacao']} (lift {ultimo['lift_relativo']:.1%}, p={ultimo['p_valor']:.4f})."}, ensure_ascii=False)

def agente(objetivo, max_passos=6):
    historico = [{"papel": "usuario", "conteudo": objetivo}]
    for passo in range(max_passos):
        saida = llm_simulado(historico)
        try:
            if "final" in json.loads(saida): return json.loads(saida)["final"], historico
        except json.JSONDecodeError:
            pass
        resultado = executar(saida)
        historico += [{"papel": "assistente", "conteudo": saida}, {"papel": "ferramenta", "conteudo": resultado}]
        print(f"passo {passo}: {saida[:90]}\n   -> {'ok' if resultado['ok'] else resultado['erro'] + ': ' + resultado['detalhe'][:70]}")
    return "Limite de passos atingido.", historico

resposta, hist = agente("Planeje e analise o teste A/B do novo botão de compra."); print("\nFINAL:", resposta)

O agente errou, leu o erro estruturado, corrigiu e seguiu. Com uma exceção em texto livre isso não aconteceria.

## Passo 8 — Automação: revisão do desenho do experimento
Uma tool que devolve problemas encontrados em uma configuração de teste, no papel de um revisor sênior.

In [ ]:
class ConfigExperimento(BaseModel):
    taxa_base: float = Field(gt=0, lt=1); mde: float = Field(gt=0); trafego_diario: int = Field(gt=0)
    duracao_dias: int = Field(gt=0); n_metricas_primarias: int = Field(default=1, ge=1); unidade: Literal["usuario", "sessao", "cluster"] = "usuario"
    tem_interacao_social: bool = False

class Revisao(BaseModel):
    aprovado: bool; n_necessario_total: int; n_previsto_total: int; problemas: list[str]

def revisar_experimento(c: ConfigExperimento) -> Revisao:
    nec = tamanho_amostra(EntradaAmostra(taxa_base=c.taxa_base, mde=c.mde)).n_total; prev = c.trafego_diario * c.duracao_dias; problemas = []
    if prev < nec: problemas.append(f"Amostra insuficiente: previsto {prev}, necessário {nec}. Aumente a duração para {math.ceil(nec / c.trafego_diario)} dias.")
    if c.duracao_dias < 7: problemas.append("Duração menor que 7 dias: não cobre o ciclo semanal.")
    if c.n_metricas_primarias > 1: problemas.append("Mais de uma métrica primária: corrija para comparações múltiplas ou escolha uma.")
    if c.tem_interacao_social and c.unidade != "cluster": problemas.append("Produto com interação social: risco de spillover. Randomize por cluster.")
    if c.unidade == "sessao": problemas.append("Randomização por sessão: o mesmo usuário pode ver as duas variantes.")
    return Revisao(aprovado=not problemas, n_necessario_total=nec, n_previsto_total=prev, problemas=problemas)

rev = revisar_experimento(ConfigExperimento(taxa_base=0.05, mde=0.05, trafego_diario=8000, duracao_dias=5, n_metricas_primarias=3, tem_interacao_social=True))
print(rev.model_dump_json(indent=1))

## Passo 9 — Do resultado estruturado à documentação
Gere o relatório do experimento em Markdown a partir dos objetos, sem digitar nenhum número à mão. Em produção, esta é a etapa que um LLM redige.

In [ ]:
def documentar(cfg: ConfigExperimento, rev: Revisao, res: Optional[ResultadoAB] = None) -> str:
    md = [f"# Experimento: revisão de desenho", f"- Taxa base: {cfg.taxa_base:.1%} | MDE: {cfg.mde:.0%} | Unidade: {cfg.unidade}",
          f"- Amostra necessária: {rev.n_necessario_total:,} | prevista: {rev.n_previsto_total:,}", f"- **Status:** {'aprovado' if rev.aprovado else 'reprovado'}"]
    md += ["", "## Problemas"] + ([f"- {p}" for p in rev.problemas] or ["- Nenhum."])
    if res:
        md += ["", "## Resultado", f"- Lift: {res.lift_relativo:.1%} | IC95% da diferença: [{res.ic95_diferenca[0]:.4f}, {res.ic95_diferenca[1]:.4f}] | p = {res.p_valor:.4f}",
               f"- **Recomendação:** {res.recomendacao}"] + [f"- Aviso: {a}" for a in res.avisos]
    return "\n".join(md)
print(documentar(ConfigExperimento(taxa_base=0.05, mde=0.05, trafego_diario=8000, duracao_dias=5, tem_interacao_social=True), rev, r))

## Fechamento da Fase 5
Especifique em `docs/` uma tool analítica do seu trabalho atual no padrão desta semana. Qual dashboard seu poderia virar uma tool para agents?